# Milestone 3 — Context Augmentation with RAG Pipelines

**Project:** Smart MCQ Solver Challenge  
**Approach:** Retrieval-Augmented Generation (RAG) using FAISS vector database  
**Metric:** MAP@3 (Mean Average Precision at 3)

In this milestone, we improve MCQ answer prediction by retrieving relevant context from a knowledge base and feeding it to the model alongside the question and options.

**The Problem with Milestone 2:**
- Sentence-transformers match prompt ↔ option by surface similarity
- They don't "know" facts — if a question asks about philosophy, the model has no knowledge to draw on
- RAG solves this by retrieving relevant knowledge and injecting it into the prompt

**Pipeline:**
1. Build a knowledge base from training data
2. Encode into FAISS vector database
3. For each question, retrieve top-k relevant passages
4. Concatenate: retrieved context + original prompt + options
5. Score options using sentence-transformer or cross-encoder

**Note:** This notebook requires GPU. Settings → Accelerator → GPU T4 x2

In [1]:
!pip install faiss-cpu sentence-transformers wandb -q

import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import wandb
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

## 1. Load the Dataset

The competition dataset contains MCQ-style questions. Each row has:
- `id` — unique question identifier
- `prompt` — the question text
- `A`, `B`, `C`, `D`, `E` — five answer option texts
- `answer` — the correct answer label (only in train set)

We load both `train.csv` (for building and evaluating models) and `test.csv` (for Kaggle submission).

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

option_cols = ['A', 'B', 'C', 'D', 'E']

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

def ap_at_3(true_label, predicted_labels):
    for i, pred in enumerate(predicted_labels[:3]):
        if pred.strip().upper() == true_label.strip().upper():
            return 1.0 / (i + 1)
    return 0.0

def map_at_3(true_labels, predicted_labels):
    return np.mean([ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)])

def map_at_3_detailed(true_labels, predicted_labels):
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    n = len(scores)
    return {
        'map3': np.mean(scores),
        'correct_at_1': sum(1 for s in scores if s == 1.0),
        'correct_at_2': sum(1 for s in scores if s == 0.5),
        'correct_at_3': sum(1 for s in scores if abs(s - 1/3) < 0.01),
        'missed': sum(1 for s in scores if s == 0.0),
        'total': n,
        'top1_acc': sum(1 for s in scores if s == 1.0) / n,
        'top3_acc': sum(1 for s in scores if s > 0) / n,
    }

def print_results(name, results):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  MAP@3:          {results['map3']:.4f}")
    print(f"  Top-1 Accuracy: {results['top1_acc']:.2%}")
    print(f"  Top-3 Accuracy: {results['top3_acc']:.2%}")
    print(f"  Correct at #1:  {results['correct_at_1']}/{results['total']}")
    print(f"  Correct at #2:  {results['correct_at_2']}/{results['total']}")
    print(f"  Correct at #3:  {results['correct_at_3']}/{results['total']}")
    print(f"  Missed:         {results['missed']}/{results['total']}")

print("Setup complete.")

Train: (2000, 8), Test: (500, 7)
Setup complete.


## 2. W&B Login

Login to Weights & Biases for experiment tracking. API key is stored as a Kaggle Secret named `WANDB_API_KEY`.

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login successful!


## 3. Build the Knowledge Base

We create knowledge entries from the training data itself. Each entry combines the question prompt with its correct answer text, giving the retrieval system both question context and factual content.

This is a **self-contained RAG** approach — no external datasets needed.

In [4]:
knowledge_entries = []
knowledge_metadata = []

for idx, row in train_df.iterrows():
    correct_answer_text = str(row[row['answer']])
    entry = f"Question: {row['prompt']} Answer: {correct_answer_text}"
    knowledge_entries.append(entry)
    knowledge_metadata.append({
        'source_idx': idx,
        'answer_label': row['answer'],
    })

print(f"Knowledge base size: {len(knowledge_entries)} entries")
print(f"\nSample entry:")
print(knowledge_entries[0][:200] + "...")

Knowledge base size: 2000 entries

Sample entry:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. Answer: Martin Heidegger believes that humans d...


## 4. Encode Knowledge Base & Build FAISS Index

**FAISS (Facebook AI Similarity Search)** enables efficient similarity search over dense vectors.

Steps:
1. Encode all knowledge entries into dense vectors using a sentence-transformer
2. Build a FAISS index from these vectors
3. At query time, encode the question and find nearest knowledge entries

In [5]:
encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Encoding knowledge base...")
kb_embeddings = encoder.encode(
    knowledge_entries,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embeddings shape: {kb_embeddings.shape}")

embedding_dim = kb_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(kb_embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} vectors")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding knowledge base...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings shape: (2000, 384)
FAISS index built with 2000 vectors


## 5. Retrieval Function

Given a query, we encode it and search FAISS for the top-k most similar knowledge entries.

**Important:** When evaluating on the training set, we exclude the question's own entry from results (leave-one-out) to avoid data leakage.

In [6]:
def retrieve_context(query, encoder, index, knowledge_entries, top_k=3, exclude_idx=None):
    """Retrieve top-k relevant knowledge entries for a given query."""
    query_vec = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    
    search_k = top_k + 1 if exclude_idx is not None else top_k
    scores, indices = index.search(query_vec, search_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if exclude_idx is not None and idx == exclude_idx:
            continue
        if len(results) >= top_k:
            break
        results.append((knowledge_entries[idx], float(score)))
    
    return results

# Test retrieval
test_query = train_df.iloc[0]['prompt']
retrieved = retrieve_context(test_query, encoder, index, knowledge_entries, top_k=3, exclude_idx=0)

print(f"Query: {test_query[:100]}...\n")
print("Retrieved context:")
for i, (entry, score) in enumerate(retrieved):
    print(f"\n  [{i+1}] Score: {score:.4f}")
    print(f"      {entry[:150]}...")

Query: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...

Retrieved context:

  [1] Score: 0.8855
      Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed option...

  [2] Score: 0.8829
      Question: Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? from the following ch...

  [3] Score: 0.8704
      Question: Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? carefully. Answer: Ma...


## 6. RAG Approach 1 — Context-Augmented Sentence Embeddings

For each question, retrieve relevant context and prepend it to the prompt. Then use sentence-transformer embeddings to compare the augmented prompt against each option.

**Augmented format:** `Context: [retrieved passages] Question: [original prompt]`

In [7]:
def predict_rag_sbert(df, encoder, index, knowledge_entries, top_k=3, is_train=True):
    """RAG prediction using augmented prompts + sentence-transformer similarity."""
    predictions = []
    
    option_embeddings = {}
    for col in option_cols:
        option_embeddings[col] = encoder.encode(
            df[col].astype(str).tolist(),
            batch_size=64, show_progress_bar=False,
            convert_to_numpy=True, normalize_embeddings=True
        )
    
    for idx in range(len(df)):
        if idx % 200 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        row = df.iloc[idx]
        prompt = row['prompt']
        
        exclude = idx if is_train else None
        retrieved = retrieve_context(prompt, encoder, index, knowledge_entries, top_k=top_k, exclude_idx=exclude)
        
        context_text = " ".join([entry for entry, score in retrieved])
        augmented_prompt = f"Context: {context_text} Question: {prompt}"
        
        prompt_vec = encoder.encode([augmented_prompt], normalize_embeddings=True)
        
        similarities = {}
        for col in option_cols:
            opt_vec = option_embeddings[col][idx].reshape(1, -1)
            similarities[col] = cosine_similarity(prompt_vec, opt_vec)[0][0]
        
        ranked = sorted(similarities, key=similarities.get, reverse=True)
        predictions.append(ranked[:3])
    
    return predictions

print("Running RAG + SBERT (top_k=3)...")
train_preds_rag_sbert_k3 = predict_rag_sbert(train_df, encoder, index, knowledge_entries, top_k=3)
results_rag_sbert_k3 = map_at_3_detailed(train_df['answer'].tolist(), train_preds_rag_sbert_k3)
print_results("RAG + SBERT (k=3)", results_rag_sbert_k3)

Running RAG + SBERT (top_k=3)...
  Processing 0/2000...
  Processing 200/2000...
  Processing 400/2000...
  Processing 600/2000...
  Processing 800/2000...
  Processing 1000/2000...
  Processing 1200/2000...
  Processing 1400/2000...
  Processing 1600/2000...
  Processing 1800/2000...

RAG + SBERT (k=3)
  MAP@3:          0.8794
  Top-1 Accuracy: 81.45%
  Top-3 Accuracy: 96.10%
  Correct at #1:  1629/2000
  Correct at #2:  193/2000
  Correct at #3:  100/2000
  Missed:         78/2000


### 6.1 Effect of Retrieval Depth (top_k)

Testing different numbers of retrieved passages. More context isn't always better — too much adds noise.

In [8]:
print("Running RAG + SBERT (top_k=1)...")
train_preds_rag_sbert_k1 = predict_rag_sbert(train_df, encoder, index, knowledge_entries, top_k=1)
results_rag_sbert_k1 = map_at_3_detailed(train_df['answer'].tolist(), train_preds_rag_sbert_k1)
print_results("RAG + SBERT (k=1)", results_rag_sbert_k1)

print("\nRunning RAG + SBERT (top_k=5)...")
train_preds_rag_sbert_k5 = predict_rag_sbert(train_df, encoder, index, knowledge_entries, top_k=5)
results_rag_sbert_k5 = map_at_3_detailed(train_df['answer'].tolist(), train_preds_rag_sbert_k5)
print_results("RAG + SBERT (k=5)", results_rag_sbert_k5)

Running RAG + SBERT (top_k=1)...
  Processing 0/2000...
  Processing 200/2000...
  Processing 400/2000...
  Processing 600/2000...
  Processing 800/2000...
  Processing 1000/2000...
  Processing 1200/2000...
  Processing 1400/2000...
  Processing 1600/2000...
  Processing 1800/2000...

RAG + SBERT (k=1)
  MAP@3:          0.8378
  Top-1 Accuracy: 75.35%
  Top-3 Accuracy: 94.20%
  Correct at #1:  1507/2000
  Correct at #2:  258/2000
  Correct at #3:  119/2000
  Missed:         116/2000

Running RAG + SBERT (top_k=5)...
  Processing 0/2000...
  Processing 200/2000...
  Processing 400/2000...
  Processing 600/2000...
  Processing 800/2000...
  Processing 1000/2000...
  Processing 1200/2000...
  Processing 1400/2000...
  Processing 1600/2000...
  Processing 1800/2000...

RAG + SBERT (k=5)
  MAP@3:          0.8852
  Top-1 Accuracy: 82.20%
  Top-3 Accuracy: 96.15%
  Correct at #1:  1644/2000
  Correct at #2:  201/2000
  Correct at #3:  78/2000
  Missed:         77/2000


## 7. RAG Approach 2 — Cross-Encoder Re-ranking

**Cross-encoders** take both texts as input simultaneously and use cross-attention to deeply compare them. This is more powerful than sentence-transformers (which encode independently) but slower.

**Our approach:**
1. Retrieve context using FAISS
2. Create augmented pairs: `(context + prompt, option)`
3. Score each pair with a cross-encoder
4. Rank by score

Cross-encoders are ideal for re-ranking a small set of candidates (5 options).

In [9]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-encoder loaded: ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded: ms-marco-MiniLM-L-6-v2


### 7.1 Cross-Encoder RAG Prediction

For each question: retrieve context → build augmented prompt → score all 5 options with cross-encoder → rank top 3.

In [10]:
def predict_rag_crossencoder(df, encoder, cross_enc, index, knowledge_entries, top_k=3, is_train=True):
    """RAG prediction using retrieved context + cross-encoder scoring."""
    predictions = []
    
    for idx in range(len(df)):
        if idx % 100 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        row = df.iloc[idx]
        prompt = row['prompt']
        
        exclude = idx if is_train else None
        retrieved = retrieve_context(prompt, encoder, index, knowledge_entries, top_k=top_k, exclude_idx=exclude)
        
        context_text = " ".join([entry for entry, score in retrieved])
        augmented_prompt = f"Context: {context_text} Question: {prompt}"
        augmented_prompt = augmented_prompt[:1500]
        
        pairs = [(augmented_prompt, str(row[col])) for col in option_cols]
        scores = cross_enc.predict(pairs)
        
        option_scores = {col: float(scores[i]) for i, col in enumerate(option_cols)}
        ranked = sorted(option_scores, key=option_scores.get, reverse=True)
        predictions.append(ranked[:3])
    
    return predictions

print("Running RAG + Cross-Encoder (top_k=3)...")
train_preds_rag_ce = predict_rag_crossencoder(
    train_df, encoder, cross_encoder, index, knowledge_entries, top_k=3
)
results_rag_ce = map_at_3_detailed(train_df['answer'].tolist(), train_preds_rag_ce)
print_results("RAG + Cross-Encoder (k=3)", results_rag_ce)

Running RAG + Cross-Encoder (top_k=3)...
  Processing 0/2000...
  Processing 100/2000...
  Processing 200/2000...
  Processing 300/2000...
  Processing 400/2000...
  Processing 500/2000...
  Processing 600/2000...
  Processing 700/2000...
  Processing 800/2000...
  Processing 900/2000...
  Processing 1000/2000...
  Processing 1100/2000...
  Processing 1200/2000...
  Processing 1300/2000...
  Processing 1400/2000...
  Processing 1500/2000...
  Processing 1600/2000...
  Processing 1700/2000...
  Processing 1800/2000...
  Processing 1900/2000...

RAG + Cross-Encoder (k=3)
  MAP@3:          0.7692
  Top-1 Accuracy: 65.20%
  Top-3 Accuracy: 91.80%
  Correct at #1:  1304/2000
  Correct at #2:  342/2000
  Correct at #3:  190/2000
  Missed:         164/2000


## 8. Baselines Without RAG

To measure the actual impact of RAG, we run the same models without retrieval augmentation.

In [11]:
# No-RAG: plain sentence-transformer
def predict_sbert_no_rag(df, encoder):
    prompt_embeddings = encoder.encode(
        df['prompt'].tolist(), batch_size=64,
        show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
    )
    option_embeddings = {}
    for col in option_cols:
        option_embeddings[col] = encoder.encode(
            df[col].astype(str).tolist(), batch_size=64,
            show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True
        )
    
    predictions = []
    for idx in range(len(df)):
        prompt_vec = prompt_embeddings[idx].reshape(1, -1)
        similarities = {}
        for col in option_cols:
            opt_vec = option_embeddings[col][idx].reshape(1, -1)
            similarities[col] = cosine_similarity(prompt_vec, opt_vec)[0][0]
        ranked = sorted(similarities, key=similarities.get, reverse=True)
        predictions.append(ranked[:3])
    return predictions

print("Running SBERT without RAG...")
train_preds_no_rag = predict_sbert_no_rag(train_df, encoder)
results_no_rag = map_at_3_detailed(train_df['answer'].tolist(), train_preds_no_rag)
print_results("SBERT without RAG", results_no_rag)

# No-RAG: plain cross-encoder
def predict_crossencoder_no_rag(df, cross_enc):
    predictions = []
    for idx in range(len(df)):
        if idx % 100 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        row = df.iloc[idx]
        pairs = [(row['prompt'], str(row[col])) for col in option_cols]
        scores = cross_enc.predict(pairs)
        option_scores = {col: float(scores[i]) for i, col in enumerate(option_cols)}
        ranked = sorted(option_scores, key=option_scores.get, reverse=True)
        predictions.append(ranked[:3])
    return predictions

print("\nRunning Cross-Encoder without RAG...")
train_preds_ce_no_rag = predict_crossencoder_no_rag(train_df, cross_encoder)
results_ce_no_rag = map_at_3_detailed(train_df['answer'].tolist(), train_preds_ce_no_rag)
print_results("Cross-Encoder without RAG", results_ce_no_rag)

Running SBERT without RAG...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


SBERT without RAG
  MAP@3:          0.4231
  Top-1 Accuracy: 26.10%
  Top-3 Accuracy: 64.70%
  Correct at #1:  522/2000
  Correct at #2:  401/2000
  Correct at #3:  371/2000
  Missed:         706/2000

Running Cross-Encoder without RAG...
  Processing 0/2000...
  Processing 100/2000...
  Processing 200/2000...
  Processing 300/2000...
  Processing 400/2000...
  Processing 500/2000...
  Processing 600/2000...
  Processing 700/2000...
  Processing 800/2000...
  Processing 900/2000...
  Processing 1000/2000...
  Processing 1100/2000...
  Processing 1200/2000...
  Processing 1300/2000...
  Processing 1400/2000...
  Processing 1500/2000...
  Processing 1600/2000...
  Processing 1700/2000...
  Processing 1800/2000...
  Processing 1900/2000...

Cross-Encoder without RAG
  MAP@3:          0.4631
  Top-1 Accuracy: 29.55%
  Top-3 Accuracy: 68.95%
  Correct at #1:  591/2000
  Correct at #2:  435/2000
  Correct at #3:  353/2000
  Missed:         621/2000


## 9. Full Comparison Table

In [12]:
comparison = pd.DataFrame({
    'Model': [
        'SBERT (no RAG)',
        'SBERT + RAG (k=1)',
        'SBERT + RAG (k=3)',
        'SBERT + RAG (k=5)',
        'Cross-Encoder (no RAG)',
        'Cross-Encoder + RAG (k=3)',
    ],
    'MAP@3': [
        results_no_rag['map3'],
        results_rag_sbert_k1['map3'],
        results_rag_sbert_k3['map3'],
        results_rag_sbert_k5['map3'],
        results_ce_no_rag['map3'],
        results_rag_ce['map3'],
    ],
    'Top-1 Acc': [
        results_no_rag['top1_acc'],
        results_rag_sbert_k1['top1_acc'],
        results_rag_sbert_k3['top1_acc'],
        results_rag_sbert_k5['top1_acc'],
        results_ce_no_rag['top1_acc'],
        results_rag_ce['top1_acc'],
    ],
    'Top-3 Acc': [
        results_no_rag['top3_acc'],
        results_rag_sbert_k1['top3_acc'],
        results_rag_sbert_k3['top3_acc'],
        results_rag_sbert_k5['top3_acc'],
        results_ce_no_rag['top3_acc'],
        results_rag_ce['top3_acc'],
    ]
})

comparison = comparison.sort_values('MAP@3', ascending=False).reset_index(drop=True)

display_df = comparison.copy()
display_df['MAP@3'] = display_df['MAP@3'].apply(lambda x: f"{x:.4f}")
display_df['Top-1 Acc'] = display_df['Top-1 Acc'].apply(lambda x: f"{x:.2%}")
display_df['Top-3 Acc'] = display_df['Top-3 Acc'].apply(lambda x: f"{x:.2%}")

print(display_df.to_string(index=False))

sbert_improvement = results_rag_sbert_k3['map3'] - results_no_rag['map3']
ce_improvement = results_rag_ce['map3'] - results_ce_no_rag['map3']
print(f"\nRAG improvement (SBERT):         {sbert_improvement:+.4f}")
print(f"RAG improvement (Cross-Encoder): {ce_improvement:+.4f}")

                    Model  MAP@3 Top-1 Acc Top-3 Acc
        SBERT + RAG (k=5) 0.8852    82.20%    96.15%
        SBERT + RAG (k=3) 0.8794    81.45%    96.10%
        SBERT + RAG (k=1) 0.8378    75.35%    94.20%
Cross-Encoder + RAG (k=3) 0.7692    65.20%    91.80%
   Cross-Encoder (no RAG) 0.4631    29.55%    68.95%
           SBERT (no RAG) 0.4231    26.10%    64.70%

RAG improvement (SBERT):         +0.4563
RAG improvement (Cross-Encoder): +0.3061


## 10. Log All Runs to W&B

In [13]:
PROJECT_NAME = "22f3002548-t22026"

experiments = [
    ("m3-sbert-no-rag", "sbert_no_rag", results_no_rag, {"retrieval": "none", "model": "MiniLM-L6-v2"}),
    ("m3-sbert-rag-k1", "sbert_rag_k1", results_rag_sbert_k1, {"retrieval": "faiss", "top_k": 1, "model": "MiniLM-L6-v2"}),
    ("m3-sbert-rag-k3", "sbert_rag_k3", results_rag_sbert_k3, {"retrieval": "faiss", "top_k": 3, "model": "MiniLM-L6-v2"}),
    ("m3-sbert-rag-k5", "sbert_rag_k5", results_rag_sbert_k5, {"retrieval": "faiss", "top_k": 5, "model": "MiniLM-L6-v2"}),
    ("m3-crossenc-no-rag", "crossencoder_no_rag", results_ce_no_rag, {"retrieval": "none", "model": "ms-marco-MiniLM-L-6-v2"}),
    ("m3-crossenc-rag-k3", "crossencoder_rag_k3", results_rag_ce, {"retrieval": "faiss", "top_k": 3, "model": "ms-marco-MiniLM-L-6-v2"}),
]

for run_name, model_label, results, extra in experiments:
    wandb.init(project=PROJECT_NAME, name=run_name, tags=["milestone3", "rag"])
    wandb.log({
        "model": model_label,
        "map3": results['map3'],
        "top1_accuracy": results['top1_acc'],
        "top3_accuracy": results['top3_acc'],
        "missed": results['missed'],
        **extra
    })
    wandb.finish()

print(f"All {len(experiments)} runs logged to W&B!")

wandb: setting up run y1nb5o7p
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260721_150015-y1nb5o7p
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run m3-sbert-no-rag
wandb: ⭐️ View project at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: 🚀 View run at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/y1nb5o7p
wandb: updating run metadata; uploading summary
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:          map3 ▁
wandb:        missed ▁
wandb: top1_accuracy ▁
wandb: top3_accuracy ▁
wandb: 
wandb: Run summary:
wandb:          map3 0.42308
wandb:        missed 706
wandb:         model MiniLM-L6-v2
wandb:     retrieval none
wandb: top1_accuracy 0.261
wandb: top3_accuracy 0.647
wandb: 
wandb: 🚀 View run m3-sbert-no-rag at: https://wandb.ai/22f3002548-dl-genai-p

All 6 runs logged to W&B!


## 11. Error Analysis — Where Does RAG Help vs Hurt?

In [14]:
scores_no_rag = [ap_at_3(t, p) for t, p in zip(train_df['answer'].tolist(), train_preds_no_rag)]
scores_rag = [ap_at_3(t, p) for t, p in zip(train_df['answer'].tolist(), train_preds_rag_sbert_k3)]

rag_helped = [(i, scores_no_rag[i], scores_rag[i]) 
              for i in range(len(scores_no_rag)) 
              if scores_rag[i] > scores_no_rag[i]]

rag_hurt = [(i, scores_no_rag[i], scores_rag[i]) 
            for i in range(len(scores_no_rag)) 
            if scores_rag[i] < scores_no_rag[i]]

print(f"RAG helped on {len(rag_helped)} questions")
print(f"RAG hurt on {len(rag_hurt)} questions")
print(f"No change on {len(scores_no_rag) - len(rag_helped) - len(rag_hurt)} questions")

print(f"\n{'='*60}")
print("EXAMPLES WHERE RAG HELPED:")
print(f"{'='*60}")
for idx, old_score, new_score in rag_helped[:3]:
    row = train_df.iloc[idx]
    print(f"\n  Q: {str(row['prompt'])[:120]}...")
    print(f"  Correct: {row['answer']}")
    print(f"  Without RAG: {' '.join(train_preds_no_rag[idx])} (score: {old_score:.3f})")
    print(f"  With RAG:    {' '.join(train_preds_rag_sbert_k3[idx])} (score: {new_score:.3f})")

print(f"\n{'='*60}")
print("EXAMPLES WHERE RAG HURT:")
print(f"{'='*60}")
for idx, old_score, new_score in rag_hurt[:3]:
    row = train_df.iloc[idx]
    print(f"\n  Q: {str(row['prompt'])[:120]}...")
    print(f"  Correct: {row['answer']}")
    print(f"  Without RAG: {' '.join(train_preds_no_rag[idx])} (score: {old_score:.3f})")
    print(f"  With RAG:    {' '.join(train_preds_rag_sbert_k3[idx])} (score: {new_score:.3f})")

RAG helped on 1347 questions
RAG hurt on 42 questions
No change on 611 questions

EXAMPLES WHERE RAG HELPED:

  Q: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo...
  Correct: B
  Without RAG: C D B (score: 0.333)
  With RAG:    B C D (score: 1.000)

  Q: What is accelerator-based light-ion fusion?...
  Correct: A
  Without RAG: B C D (score: 0.000)
  With RAG:    A C D (score: 1.000)

  Q: Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting in e...
  Correct: C
  Without RAG: B A D (score: 0.000)
  With RAG:    B C A (score: 0.500)

EXAMPLES WHERE RAG HURT:

  Q: Choose the correct answer: What is the definition of an improper rotation?...
  Correct: B
  Without RAG: B E A (score: 1.000)
  With RAG:    D C B (score: 0.333)

  Q: Which of the following is correct? What is the Ramsauer-Townsend effect? carefully....
  Correct: B
  Without RAG: B D C (

## 12. Generate Kaggle Submission

In [15]:
all_results = {
    'sbert_rag_k1': results_rag_sbert_k1['map3'],
    'sbert_rag_k3': results_rag_sbert_k3['map3'],
    'sbert_rag_k5': results_rag_sbert_k5['map3'],
    'crossenc_rag_k3': results_rag_ce['map3'],
    'crossenc_no_rag': results_ce_no_rag['map3'],
    'sbert_no_rag': results_no_rag['map3'],
}
best_approach = max(all_results, key=all_results.get)
print(f"Best approach: {best_approach} with MAP@3 = {all_results[best_approach]:.4f}")

print(f"\nGenerating test predictions...")

if 'crossenc' in best_approach and 'rag' in best_approach:
    test_preds = predict_rag_crossencoder(test_df, encoder, cross_encoder, index, knowledge_entries, top_k=3, is_train=False)
elif 'crossenc' in best_approach:
    test_preds = predict_crossencoder_no_rag(test_df, cross_encoder)
elif 'rag' in best_approach:
    k = int(best_approach.split('k')[1]) if 'k' in best_approach else 3
    test_preds = predict_rag_sbert(test_df, encoder, index, knowledge_entries, top_k=k, is_train=False)
else:
    test_preds = predict_sbert_no_rag(test_df, encoder)

submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': [' '.join(pred) for pred in test_preds]
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

submission.to_csv('submission.csv', index=False)
print("\nSaved to submission.csv")

Best approach: sbert_rag_k5 with MAP@3 = 0.8852

Generating test predictions...
  Processing 0/500...
  Processing 200/500...
  Processing 400/500...

Submission shape: (500, 2)
   id prediction
0   1      B A D
1   2      B C D
2   3      B C D
3   4      E C A
4   5      C D A

Saved to submission.csv


In [16]:
submission = pd.read_csv('submission.csv')
print(f"Submission shape: {submission.shape}")
print(f"Columns: {list(submission.columns)}")
print(f"Sample predictions:")
print(submission.head(10))

assert submission.shape[0] == len(test_df), "Row count mismatch!"
assert all(len(p.split()) == 3 for p in submission['prediction']), "All predictions must have exactly 3 labels!"
print(f"\n✓ All {len(submission)} rows have exactly 3 predictions. Ready to submit!")

Submission shape: (500, 2)
Columns: ['id', 'prediction']
Sample predictions:
   id prediction
0   1      B A D
1   2      B C D
2   3      B C D
3   4      E C A
4   5      C D A
5   6      D A E
6   7      E D C
7   8      A C D
8   9      C E A
9  10      B C E

✓ All 500 rows have exactly 3 predictions. Ready to submit!
